# Section 7 · Observability 1 — Console Tracing (no external services) (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

Strands emits OpenTelemetry (OTel) spans for every agent invocation, reasoning cycle, model call and tool call. The simplest way to see them: print them to the console.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

In [ ]:
from strands import Agent
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator

# Print every span to stdout
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

agent = Agent(model=model, system_prompt="You are a helpful assistant.", tools=[calculator], callback_handler=None)

response = agent("What is 15% of 2,847? Then explain in two sentences why tracing matters for AI agents.")
print("\n\nANSWER:\n", response)

In the output above, look for:
- an **`invoke_agent`** span (top level) with total token counts
- **`execute_event_loop_cycle`** spans — one per reasoning loop
- **`chat`** spans — each model call, with input/output tokens and latency
- **`execute_tool calculator`** — the tool call with its parameters and result